# SymPhase tutorial 1 — magnetic symmetry and symmetry-allowed conductivity tensors

**Goal:** from a finished VASP calculation (`POSCAR`, `OUTCAR`, `OSZICAR`) obtain

* the crystal space group and the **magnetic space group** (BNS/OG), magnetic point group, MSG type,
* which symmetry elements are present: inversion **P**, time reversal **T**, **PT**, **T·τ**, mirrors, **M·T**, **C<sub>n</sub>·T** …,
* the symmetry-allowed shape of the **charge** σ<sub>ij</sub>, **spin** σ<sup>k</sup><sub>ij</sub> and **orbital** σ<sup>k</sup><sub>ij</sub> conductivity tensors, split into T-even and T-odd parts.

Command line equivalent: `symphase tensors` inside a VASP folder.

## 1. Theory in five lines

**Neumann's principle:** a response tensor must be invariant under every operation of the (magnetic) point group.
For an operation $g=\{R|\tau\}$ or $g'=\{R|\tau\}\mathcal{T}$ acting on a tensor $\chi$:

$$\chi^{\text{T-even}} = D(R)\,\chi^{\text{T-even}},\qquad \chi^{\text{T-odd}} = \theta_g\,D(R)\,\chi^{\text{T-odd}},\qquad \theta_g=\begin{cases}+1 & g\\ -1 & g'\end{cases}$$

| response | equation | $D(R)$ |
|---|---|---|
| charge | $J_i=\sigma_{ij}E_j$ | $R\otimes R$ (polar rank 2) |
| spin | $J^{S_k}_i=\sigma^k_{ij}E_j$ | $\det(R)\,R\otimes R\otimes R$ (axial rank 3) |
| orbital | $J^{L_k}_i=\sigma^k_{ij}E_j$ | same as spin ( $\mathbf L$ is axial and T-odd) |

For the charge conductivity Onsager's relation $\sigma_{ij}(\mathbf M)=\sigma_{ji}(-\mathbf M)$ is imposed in addition:
T-even part symmetric (Drude), T-odd part antisymmetric (**anomalous Hall**, Hall vector $h_k$ with $\sigma^{odd}_{ij}=\epsilon_{ijk}h_k$).

SymPhase builds the linear constraints for all operations, finds the null space numerically (SVD) and prints each tensor in terms of its independent components.

In [1]:
# run this first (works from the repository's notebooks/ folder)
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # use the local package if not pip-installed
EX = os.path.abspath("../examples")
%matplotlib inline

## 2. Non-magnetic fcc Pt — the textbook spin Hall metal

In [2]:
from symphase import analyze_tensors
res = analyze_tensors(f"{EX}/vasp/fcc_Pt", quiet=False, no_ops=True, tensors="charge,spin")


SymPhase 1.0.0  —  symmetry & linear-response tensor analysis
  POSCAR  : /home/claude/symphase/examples/vasp/fcc_Pt/POSCAR   (Pt)
  OUTCAR  : /home/claude/symphase/examples/vasp/fcc_Pt/OUTCAR
  OSZICAR : /home/claude/symphase/examples/vasp/fcc_Pt/OSZICAR

-- Calculation ---------------------------------------------------------------
  VASP version         : vasp.6.4.2
  ISPIN                : 1
  LNONCOLLINEAR/LSORBIT: False / True
  SAXIS                : ( 0.000,  0.000,  1.000)
  E-fermi              : 5.1234 eV
  TOTEN (last)         : -12.345678 eV
  OSZICAR              : 1 ionic step(s), F = -12.345678 eV, E0 = -12.345000 eV

-- Structure -----------------------------------------------------------------
  lattice vectors (Å):
    a = ( 0.000000,  1.960000,  1.960000)
    b = ( 1.960000,  0.000000,  1.960000)
    c = ( 1.960000,  1.960000,  0.000000)
  a b c = 2.77186 2.77186 2.77186 Å    α β γ = 60.000 60.000 60.000°    V = 15.0591 Å^3
  moments source: none -> non-magnetic
  


-- SPIN CONDUCTIVITY  J^{S_k}_i = σ^k_ij E_j   (i: flow, j: field, k: spin polarisation) 

  T-even (intrinsic SHE):  1 independent component(s)
    independent: σ^x_yz
    k = x  (S_x polarisation):
      ⎡   0        0        0   ⎤
      ⎢   0        0      σ^x_yz⎥
      ⎣   0     -σ^x_yz     0   ⎦
    k = y  (S_y polarisation):
      ⎡   0        0     -σ^x_yz⎤
      ⎢   0        0        0   ⎥
      ⎣ σ^x_yz     0        0   ⎦
    k = z  (S_z polarisation):
      ⎡   0      σ^x_yz     0   ⎤
      ⎢-σ^x_yz     0        0   ⎥
      ⎣   0        0        0   ⎦
    transverse pairs (k:ij)  antisymmetric σ^k_ij=-σ^k_ji (SHE-like): x:yz, y:xz, z:xy
                             symmetric σ^k_ij=+σ^k_ji (spin-splitter-like): none
                             independent ij, ji (mixed): none
    k ⟂ i ⟂ j (conventional geometry) nonzero : σ^x_yz, σ^x_zy, σ^y_xz, σ^y_zx, σ^z_xy, σ^z_yx
    unconventional geometry nonzero          : none
    allowed polarisations for transverse flow i ⟂ fiel

**Read the output:** grey group `Fm-3m1'` (type II, T is a symmetry) ⇒ every T-odd tensor vanishes.
The spin Hall tensor has a single independent component, $\sigma^z_{xy}=-\sigma^z_{yx}=\sigma^x_{yz}=\dots$ — the familiar $\epsilon_{ijk}$ form.

## 3. Ferromagnetic bcc Fe with **m ∥ z** (SOC, non-collinear OUTCAR)

In [3]:
res = analyze_tensors(f"{EX}/vasp/bcc_Fe_m001", tensors="charge,spin")
m = res["magnetic"]
print("magnetic space group:", m["bns_symbol"], m["bns_number"], "| MPG:", m["magnetic_point_group"])
print("inversion:", m["inversion"], "| PT:", m["PT"], "| T:", m["time_reversal"])
print("Hall vector h =", res["tensors"]["charge"]["hall_vector"])
print("T-odd (magnetic) spin-conductivity components:", res["tensors"]["spin"]["n_odd"])

magnetic space group: I4/mm'm' 139.537 | MPG: 4/mm'm'
inversion: True | PT: False | T: False
Hall vector h = ['0', '0', 'σ_xy']
T-odd (magnetic) spin-conductivity components: 4


The full report is in `res["report"]`. Print just the T-odd spin block:

In [4]:
rep = res["report"]
print(rep[rep.index("T-odd (magnetic SHE"):rep.index("allowed polarisations", rep.index("T-odd (magnetic SHE"))])

T-odd (magnetic SHE, ∝ 1/Γ):  4 independent component(s)
    independent: σ^x_xz, σ^x_zx, σ^z_xx, σ^z_zz
    k = x  (S_x polarisation):
      ⎡  0       0     σ^x_xz⎤
      ⎢  0       0       0   ⎥
      ⎣σ^x_zx    0       0   ⎦
    k = y  (S_y polarisation):
      ⎡  0       0       0   ⎤
      ⎢  0       0     σ^x_xz⎥
      ⎣  0     σ^x_zx    0   ⎦
    k = z  (S_z polarisation):
      ⎡σ^z_xx    0       0   ⎤
      ⎢  0     σ^z_xx    0   ⎥
      ⎣  0       0     σ^z_zz⎦
    transverse pairs (k:ij)  antisymmetric σ^k_ij=-σ^k_ji (SHE-like): none
                             symmetric σ^k_ij=+σ^k_ji (spin-splitter-like): none
                             independent ij, ji (mixed): x:xz, y:yz
    k ⟂ i ⟂ j (conventional geometry) nonzero : none
    unconventional geometry nonzero          : σ^x_xz, σ^x_zx, σ^y_yz, σ^y_zy, σ^z_xx, σ^z_yy, σ^z_zz
    


## 4. RuO₂ — an altermagnet: the Néel vector decides everything

Same crystal (`P4_2/mnm`), three Néel-vector orientations. Collinear moments can be rotated with `mdir`.

In [5]:
cases = {"Néel ∥ [001]": dict(directory=f"{EX}/vasp/RuO2_Neel001_noSOC"),
         "Néel ∥ [100]": dict(directory=f"{EX}/vasp/RuO2_Neel100_SOC"),
         "Néel ∥ [110]": dict(directory=f"{EX}/vasp/RuO2_Neel001_noSOC", mdir=(1, 1, 0))}
print(f"{'case':<14s} {'MSG':<14s} {'MPG':<9s} AHE   Hall vector")
for name, kw in cases.items():
    r = analyze_tensors(tensors="charge,spin", **kw)
    c = r["tensors"]["charge"]
    print(f"{name:<14s} {r['magnetic']['bns_symbol']:<14s} {r['magnetic']['magnetic_point_group']:<9s} "
          f"{'yes' if c['ahe_allowed'] else 'no ':<5s} {c['hall_vector']}")

case           MSG            MPG       AHE   Hall vector
Néel ∥ [001]   P4_2'/mnm'     4'/mmm'   no    None
Néel ∥ [100]   Pnn'm'         mm'm'     yes   ['0', '-σ_xz', '0']
Néel ∥ [110]   Cmm'm'         mm'm'     yes   ['-σ_xz', '-σ_xz', '0']


* **[001]**: `P4_2'/mnm'` — AHE forbidden, but the T-odd spin tensor contains $\sigma^z_{xy}=+\sigma^z_{yx}$: the **spin-splitter** (symmetric) form.
* **[100]** and **[110]**: lower symmetry, anomalous (crystal) Hall effect allowed.

Check the spin-splitter form yourself:

In [6]:
r = analyze_tensors(f"{EX}/vasp/RuO2_Neel001_noSOC", tensors="spin")
odd = r["tensors"]["spin"]["odd"]
print("σ^z_xy =", odd["σ^z_xy"], "   σ^z_yx =", odd["σ^z_yx"])

σ^z_xy = σ^z_xy    σ^z_yx = σ^z_xy


## 5. Reading the symmetry elements and operations

`res["magnetic"]` contains the lists of mirrors, M·T planes, rotation axes and every operation (rotation matrix in lattice and Cartesian coordinates, translation, time reversal).

In [7]:
m = analyze_tensors(f"{EX}/vasp/RuO2_Neel100_SOC")["magnetic"]
print("mirror planes      :", *m["mirrors"], sep="\n   ")
print("mirror planes × T  :", *m["mirrors_T"], sep="\n   ")
print("rotations × T      :", *m["rotations_T"], sep="\n   ")
print("number of operations:", m["n_operations"])

mirror planes      :
   glide plane ⟂ ( 0.000,  1.000,  0.000) = plane (010)
mirror planes × T  :
   mirror plane ⟂ ( 0.000,  0.000,  1.000) = plane (001)
   glide plane ⟂ ( 1.000,  0.000,  0.000) = plane (100)
rotations × T      :
   2-fold rotation ∥ ( 0.000,  0.000,  1.000) = [001]
   2-fold screw ∥ ( 1.000,  0.000,  0.000) = [100]
number of operations: 8


## 6. Command line, JSON and useful options

```bash
cd my_vasp_run/
symphase tensors                          # POSCAR/OUTCAR/OSZICAR of this folder
symphase tensors -p CONTCAR --mmin 0.1    # final structure, ignore |m| < 0.1 μB
symphase tensors --mdir 1 1 0             # rotate collinear moments to [110]
symphase tensors --magmom "0 0 3  0 0 -3  4*0"   # override moments (3N Cartesian)
symphase tensors --frame std              # tensors in the standardized frame
symphase tensors --json res.json --out report.txt
```

**Caveats**
* The magnetic group couples spin and lattice (relativistic). Without SOC the true symmetry is a larger *spin group*.
* Small induced moments can lower the symmetry: tune `mmin` / `mag_symprec`.
* Tensors are given in the POSCAR Cartesian frame unless `frame="std"`.

## 7. Exercise

Rotate the bcc-Fe moment to [111] with `magmom="0.577 0.577 0.577  0.577 0.577 0.577"` (scaled by 2.2 if you like) and determine
(i) the magnetic point group, (ii) the direction of the Hall vector, (iii) the number of independent T-odd spin-conductivity components.
Then continue with **04_quiz_symmetry_tensors.ipynb**.

In [8]:
r = analyze_tensors(f"{EX}/vasp/bcc_Fe_m001", magmom="1.27 1.27 1.27 1.27 1.27 1.27", tensors="charge,spin")
print(r["magnetic"]["bns_symbol"], r["magnetic"]["magnetic_point_group"], r["tensors"]["charge"]["hall_vector"],
      r["tensors"]["spin"]["n_odd"])

R-3m' -3m' ['σ_xy', 'σ_xy', 'σ_xy'] 5
